# **Video Processing Week 2: Analisis Manusia (Wajah & Tubuh) dengan MediaPipe**

Minggu ini kita akan mempelajari **analisis manusia dalam video** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking fitur wajah serta pose tubuh manusia secara real-time dengan akurasi tinggi.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan model AI (via MediaPipe) untuk mengekstrak fitur wajah dan tubuh manusia secara real-time
- Mendeteksi wajah dan 468 titik landmark pada wajah dengan presisi tinggi
- Mengimplementasikan aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR)
- Mendeteksi 33 titik landmark pada tubuh manusia (skeleton/pose detection)
- Membangun aplikasi computer vision praktis untuk analisis gerakan manusia

**Topik Praktik:**
- **Pengenalan MediaPipe**: Setup dan penggunaan library MediaPipe untuk solusi AI real-time
- **Facial Landmark Detection**: Deteksi 468 titik wajah dan aplikasi deteksi kedipan mata
- **Pose Landmark Detection**: Deteksi skeleton tubuh dan analisis sudut sendi untuk deteksi gerakan

> *This module is inspired by the development of last semester’s materials.* 

> **Versi modifikasi:** visualisasi wajah, landmark, kedipan, dan pose dibuat sedikit berbeda menggunakan panel status, background blur, dan perubahan threshold ringan.

## **Pengenalan MediaPipe**

MediaPipe adalah framework open-source dari Google yang dirancang untuk membangun pipeline pemrosesan media secara real-time, seperti visi komputer dan pengolahan audio. MediaPipe menyediakan model deteksi wajah yang ringan, akurat, dan cepat, yang dapat digunakan baik untuk gambar statis maupun video streaming real-time.

*Sebelum lanjut, kita coba import library yang akan kita butuhkan dulu*

**⚠️ PENTING: Jika anda menggunakan library opencv-contrib-python dari materi minggu lalu, pastikan menggunakan versi 4.11.0.86 untuk menghindari masalah kompatibilitas dengan Mediapipe.**

In [ ]:
# pip install opencv-python numpy matplotlib mediapipe ipykernel
# atau
# pip install opencv-contrib-python==4.11.0.86 numpy matplotlib mediapipe ipykernel

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

try:
    import mediapipe as mp
    MEDIAPIPE_AVAILABLE = True
except Exception as e:
    mp = None
    MEDIAPIPE_AVAILABLE = False
    print("MediaPipe belum tersedia. Install dulu mediapipe jika ingin menjalankan bagian kamera/landmark.")
    print("Detail:", e)

# Mode aman untuk Run All. Ubah True hanya jika ingin membuka kamera/window manual.
RUN_INTERACTIVE = False
MAX_FRAMES = 180


def draw_status_panel(frame, text, color=(255, 255, 255)):
    """Panel sederhana agar keterangan terlihat rapi di atas video."""
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], 55), (0, 0, 0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0)
    cv2.putText(frame, text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
    return frame


**Penjelasan Fungsi Penting:**
* **cv2** (OpenCV): Library utama untuk semua operasi CV: membaca/menulis video, konversi warna, filter, dan object tracking.
* **numpy**: Pondasi untuk komputasi numerik, digunakan untuk memanipulasi frame video (yang merupakan array).
* **matplotlib**: Digunakan untuk menampilkan gambar atau frame video di dalam output cell Python Notebook.

### Menggunakan Mediapipe Real-time Face Mesh Detection

In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, cell face detection dilewati.')
else:
    PATH_VIDEO = os.path.join(os.getcwd(), 'data', 'man_walking.mp4')

    # Inisialisasi MediaPipe Face Detection
    mp_face = mp.solutions.face_detection
    mp_draw = mp.solutions.drawing_utils
    face_det = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.55)

    cap = cv2.VideoCapture(PATH_VIDEO)
    if not RUN_INTERACTIVE:
        print("Cell face detection video dilewati agar Run All aman. Ubah RUN_INTERACTIVE=True untuk mencoba.")
    elif not cap.isOpened():
        print(f"Gagal membuka video: {PATH_VIDEO}")
    else:
        frame_count = 0
        while frame_count < MAX_FRAMES:
            ok, frame = cap.read()
            if not ok:
                break

            height, width = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = face_det.process(rgb)

            # MODIFIKASI: area di luar wajah diberi blur ringan agar fokus ke wajah
            display = cv2.GaussianBlur(frame, (15, 15), 0)
            if result.detections:
                for det in result.detections:
                    rel_box = det.location_data.relative_bounding_box
                    x = max(0, int(rel_box.xmin * width))
                    y = max(0, int(rel_box.ymin * height))
                    bw = max(0, min(int(rel_box.width * width), width - x))
                    bh = max(0, min(int(rel_box.height * height), height - y))
                    display[y:y+bh, x:x+bw] = frame[y:y+bh, x:x+bw]
                    cv2.rectangle(display, (x, y), (x + bw, y + bh), (0, 200, 255), 2)
                    if det.score:
                        cv2.putText(display, f"Face {det.score[0]:.2f}", (x, max(20, y - 8)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2, cv2.LINE_AA)

            draw_status_panel(display, "Face Detection + Background Blur", (0, 200, 255))
            cv2.imshow("MediaPipe Face Detection Modified", display)
            frame_count += 1
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()


## Facial Landmark Detection

MediaPipe Face Mesh dapat mendeteksi hingga 478 titik landmark pada wajah manusia dengan presisi tinggi, memungkinkan analisis detail fitur wajah seperti mata, hidung, mulut, dan kontur wajah untuk berbagai aplikasi seperti deteksi emosi, tracking mata, dan augmented reality.

Berikut adalah list landmark beserta posisinya: https://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png

### Deteksi wajah dan 468 titik landmark pada wajah

In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, cell face mesh dilewati.')
else:
    # Inisialisasi MediaPipe Face Mesh dengan visualisasi kontur yang lebih ringan
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.55,
        min_tracking_confidence=0.55
    )

    if not RUN_INTERACTIVE:
        print("Cell face mesh webcam dilewati agar Run All aman.")
    else:
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Gagal membuka webcam")
        else:
            print("Tekan 'q' untuk keluar")
            frame_count = 0
            while frame_count < MAX_FRAMES:
                ret, frame = cap.read()
                if not ret:
                    print("Gagal membaca frame dari webcam")
                    break

                frame = cv2.flip(frame, 1)
                height, width = frame.shape[:2]
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = face_mesh.process(rgb)

                if results.multi_face_landmarks:
                    for face_landmarks in results.multi_face_landmarks:
                        # MODIFIKASI: tampilkan kontur dan iris saja agar tidak terlalu penuh
                        mp_drawing.draw_landmarks(
                            image=frame,
                            landmark_list=face_landmarks,
                            connections=mp_face_mesh.FACEMESH_CONTOURS,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                        )
                        mp_drawing.draw_landmarks(
                            image=frame,
                            landmark_list=face_landmarks,
                            connections=mp_face_mesh.FACEMESH_IRISES,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_iris_connections_style()
                        )

                draw_status_panel(frame, "Face Mesh: Contour + Iris Mode")
                cv2.imshow("MediaPipe Face Mesh - Modified", frame)
                frame_count += 1
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

        cap.release()
        cv2.destroyAllWindows()
    face_mesh.close()


### Aplikasi sederhana: Deteksi Kedipan Mata (menggunakan rasio aspek mata / Eye Aspect Ratio)

Aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR) bekerja dengan menghitung rasio antara jarak vertikal dan horizontal mata dari landmark wajah. Berikut tahapan implementasinya:

**Tahapan Implementasi:**

1. **Ekstraksi Koordinat Mata**: Mengambil 6 titik landmark khusus untuk setiap mata (kiri dan kanan) dari 468 landmark wajah MediaPipe
2. **Perhitungan EAR**: Menghitung Eye Aspect Ratio menggunakan rumus: `EAR = (|p2-p6| + |p3-p5|) / (2 * |p1-p4|)` dimana p1-p6 adalah 6 titik landmark mata
3. **Threshold Detection**: Membandingkan nilai EAR dengan threshold (biasanya ~0.25) - jika EAR di bawah threshold berarti mata tertutup
4. **Frame Counting**: Menghitung berapa frame berturut-turut mata tertutup untuk menghindari false positive
5. **Blink Counter**: Increment counter kedipan ketika mata kembali terbuka setelah tertutup dalam durasi yang wajar

**Kegunaan**: Aplikasi ini berguna untuk sistem monitoring kantuk pengemudi, kontrol perangkat hands-free, atau analisis perhatian dalam pembelajaran online.

**Nilai EAR Umum**
- **Mata Terbuka**: ~0.3 - 0.4
- **Mata Tertutup**: ~0.2 - 0.3

**Pemilihan Landmark**
Kode ini menggunakan indeks landmark MediaPipe Face Mesh yang spesifik:
- **Mata Kanan**: [33, 159, 158, 133, 153, 145]
- **Mata Kiri**: [362, 380, 374, 263, 386, 385]

In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, konfigurasi EAR dilewati.')
else:
    # Fungsi untuk menghitung Eye Aspect Ratio (EAR)
    def calculate_ear(eye_landmarks):
        vertical_1 = np.linalg.norm(eye_landmarks[1] - eye_landmarks[5])
        vertical_2 = np.linalg.norm(eye_landmarks[2] - eye_landmarks[4])
        horizontal = np.linalg.norm(eye_landmarks[0] - eye_landmarks[3])
        ear = (vertical_1 + vertical_2) / (2.0 * horizontal)
        return ear


    def draw_eye_polygon(frame, eye_points, color):
        """MODIFIKASI: mata digambar sebagai polygon tipis agar lebih jelas."""
        pts = np.array(eye_points, dtype=np.int32)
        cv2.polylines(frame, [pts], True, color, 2, cv2.LINE_AA)
        for point in pts:
            cv2.circle(frame, tuple(point), 2, color, -1)

    LEFT_EYE_INDEXES = [33, 160, 158, 133, 153, 144]
    RIGHT_EYE_INDEXES = [362, 385, 387, 263, 373, 380]

    # MODIFIKASI: threshold dibuat sedikit lebih sensitif dan butuh 3 frame berturut-turut
    EAR_THRESHOLD = 0.22
    CONSECUTIVE_FRAMES = 3

    blink_counter = 0
    total_blinks = 0
    recent_ear = []

    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.55,
        min_tracking_confidence=0.55
    )


In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, deteksi kedipan dilewati.')
else:
    if not RUN_INTERACTIVE:
        print("Cell deteksi kedipan webcam dilewati agar Run All aman.")
    else:
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Gagal membuka webcam")
        else:
            while True:
                ret, frame = cap.read()
                if not ret:
                    print("Gagal membaca frame dari webcam")
                    break

                frame = cv2.flip(frame, 1)
                height, width = frame.shape[:2]
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = face_mesh.process(rgb)

                if results.multi_face_landmarks:
                    for face_landmarks in results.multi_face_landmarks:
                        left_eye, right_eye = [], []
                        for idx in LEFT_EYE_INDEXES:
                            landmark = face_landmarks.landmark[idx]
                            left_eye.append(np.array([int(landmark.x * width), int(landmark.y * height)]))
                        for idx in RIGHT_EYE_INDEXES:
                            landmark = face_landmarks.landmark[idx]
                            right_eye.append(np.array([int(landmark.x * width), int(landmark.y * height)]))

                        left_ear = calculate_ear(left_eye)
                        right_ear = calculate_ear(right_eye)
                        avg_ear = (left_ear + right_ear) / 2.0

                        # MODIFIKASI: EAR dirata-ratakan 5 frame agar status tidak terlalu berkedip-kedip
                        recent_ear.append(avg_ear)
                        recent_ear[:] = recent_ear[-5:]
                        smooth_ear = float(np.mean(recent_ear))

                        if smooth_ear < EAR_THRESHOLD:
                            blink_counter += 1
                        else:
                            if blink_counter >= CONSECUTIVE_FRAMES:
                                total_blinks += 1
                            blink_counter = 0

                        status = "BERKEDIP" if smooth_ear < EAR_THRESHOLD else "TERBUKA"
                        color = (0, 80, 255) if status == "BERKEDIP" else (0, 220, 120)
                        draw_eye_polygon(frame, left_eye, color)
                        draw_eye_polygon(frame, right_eye, color)
                        draw_status_panel(frame, f"EAR: {smooth_ear:.2f} | Status: {status} | Total: {total_blinks}", color)

                cv2.imshow("Deteksi Kedipan Mata - Smooth EAR", frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    break

        cap.release()
        cv2.destroyAllWindows()
    face_mesh.close()
    print(f"Total kedipan terdeteksi: {total_blinks}")


## Pose Landmark Detection

MediaPipe Pose Landmark Detection mendeteksi dan melacak 33 titik landmark pada tubuh manusia. memungkinkan komputer untuk "melihat" dan memahami postur serta gerakan tubuh manusia.

**33 Titik Landmark Tubuh:**
- **Wajah**: Hidung, mata kiri/kanan, telinga kiri/kanan (5 titik)
- **Tubuh Atas**: Bahu, siku, pergelangan tangan kiri/kanan (6 titik) 
- **Tubuh Tengah**: Pinggul kiri/kanan (2 titik)
- **Kaki**: Pinggul, lutut, pergelangan kaki, tumit, ujung kaki kiri/kanan (20 titik)

**Kegunaan Praktis:**
- **Analisis Olahraga**: Mengukur teknik gerakan atlet, mendeteksi postur yang salah
- **Fitness Apps**: Menghitung repetisi push-up, squat, atau latihan lainnya
- **Rehabilitasi Medis**: Monitoring progress pasien fisioterapi
- **Game & AR**: Kontrol karakter game menggunakan gerakan tubuh
- **Analisis Ergonomi**: Evaluasi postur kerja untuk mencegah cedera

### Deteksi 33 titik landmark pada tubuh manusia (skeleton)

In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, pose landmark dilewati.')
else:
    # Inisialisasi MediaPipe Pose dengan skeleton lebih tebal dan panel status
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.55, min_tracking_confidence=0.55)
    mp_drawing = mp.solutions.drawing_utils

    if not RUN_INTERACTIVE:
        print("Cell pose landmark webcam dilewati agar Run All aman.")
    else:
        cap = cv2.VideoCapture(0)
        frame_count = 0
        while cap.isOpened() and frame_count < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret:
                print("Gagal mengambil frame. Keluar...")
                break

            frame = cv2.flip(frame, 1)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(frame_rgb)

            if results.pose_landmarks:
                # MODIFIKASI: garis skeleton dibuat lebih tebal dan titik lebih kecil
                mp_drawing.draw_landmarks(
                    frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2, circle_radius=2),
                    connection_drawing_spec=mp_drawing.DrawingSpec(color=(0, 200, 255), thickness=3)
                )
                draw_status_panel(frame, "Pose Detected - Thick Skeleton", (0, 200, 255))
            else:
                draw_status_panel(frame, "Pose belum terdeteksi", (0, 0, 255))

            cv2.imshow('Pose Landmark Detection Modified', frame)
            frame_count += 1
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
    pose.close()


### Aplikasi sederhana: Menghitung Sudut Siku Kanan

In [ ]:
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    angle = np.degrees(np.arccos(cosine_angle))
    return angle


def classify_elbow_angle(angle):
    """MODIFIKASI: memberi label sederhana sesuai besar sudut siku."""
    if angle < 70:
        return "Menekuk kuat"
    if angle < 130:
        return "Menekuk sedang"
    return "Hampir lurus"


In [ ]:
if not MEDIAPIPE_AVAILABLE:
    print('MediaPipe belum tersedia, sudut siku dilewati.')
else:
    BAHU_KANAN, SIKU_KANAN, TELAPAK_KANAN = 12, 14, 16

    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1,
                        enable_segmentation=False, min_detection_confidence=0.55,
                        min_tracking_confidence=0.55)

    if not RUN_INTERACTIVE:
        print("Cell sudut siku kanan dilewati agar Run All aman.")
    else:
        cap = cv2.VideoCapture(0)
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break

            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            if results.pose_landmarks:
                lm = results.pose_landmarks.landmark
                bahu = [lm[BAHU_KANAN].x * w, lm[BAHU_KANAN].y * h]
                siku = [lm[SIKU_KANAN].x * w, lm[SIKU_KANAN].y * h]
                telapak = [lm[TELAPAK_KANAN].x * w, lm[TELAPAK_KANAN].y * h]
                angle = calculate_angle(bahu, siku, telapak)
                label = classify_elbow_angle(angle)

                # MODIFIKASI: warna status berubah berdasarkan besar sudut
                color = (0, 255, 0) if angle > 130 else (0, 200, 255) if angle > 70 else (0, 80, 255)
                mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                cv2.circle(frame, tuple(np.int32(siku)), 10, color, -1)
                draw_status_panel(frame, f"Sudut siku: {angle:.1f} derajat | {label}", color)

            cv2.imshow("Right Elbow Angle - Modified", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
    pose.close()


### 🔎 Eksplorasi

- Cobalah mendeteksi bahu kiri
- Cobalah mendeteksi pose tubuh (berdiri, jongkok, tidur)

## **Ringkasan Materi: Video Processing Week 2 - Analisis Manusia dengan MediaPipe**

### **🎯 Tujuan Pembelajaran**
Mempelajari analisis manusia dalam video menggunakan teknologi AI melalui MediaPipe untuk deteksi dan tracking fitur wajah serta pose tubuh manusia secara real-time dengan akurasi tinggi.

### **📚 Materi yang Dipelajari**

#### **1. MediaPipe Framework**
- Framework open-source dari Google untuk pemrosesan media real-time
- Menyediakan model deteksi wajah yang ringan, akurat, dan cepat
- Dapat digunakan untuk gambar statis maupun video streaming real-time

#### **2. Facial Landmark Detection**
- **Deteksi Wajah**: Menggunakan MediaPipe Face Detection dengan bounding box
- **468 Titik Landmark**: MediaPipe Face Mesh mendeteksi hingga 478 titik landmark pada wajah
- **Aplikasi Praktis**: Deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR)
    - Rumus EAR: `(|p2-p6| + |p3-p5|) / (2 * |p1-p4|)`
    - Threshold EAR: ~0.2-0.3 (mata tertutup), ~0.3-0.4 (mata terbuka)

#### **3. Pose Landmark Detection**
- **33 Titik Landmark**: Deteksi dan pelacakan titik-titik kunci pada tubuh manusia
- **Distribusi Landmark**:
    - Wajah: 5 titik (hidung, mata, telinga)
    - Tubuh atas: 6 titik (bahu, siku, pergelangan tangan)
    - Tubuh tengah: 2 titik (pinggul)
    - Kaki: 20 titik (pinggul hingga ujung kaki)
- **Aplikasi**: Analisis sudut siku untuk deteksi gerakan

### **💡 Aplikasi Praktis**
- **Monitoring Kesehatan**: Deteksi kantuk pengemudi melalui kedipan mata
- **Fitness & Olahraga**: Analisis teknik gerakan dan penghitungan repetisi
- **Rehabilitasi Medis**: Monitoring progress pasien fisioterapi
- **Augmented Reality**: Kontrol perangkat menggunakan gerakan tubuh
- **Analisis Ergonomi**: Evaluasi postur kerja untuk pencegahan cedera

### **🔧 Library yang Digunakan**
- **OpenCV**: Operasi computer vision dasar
- **MediaPipe**: Model AI untuk deteksi wajah dan pose
- **NumPy**: Komputasi numerik dan manipulasi array
- **Matplotlib**: Visualisasi hasil

### **📈 Hasil Pembelajaran**
Mampu membangun aplikasi computer vision yang dapat:
1. Mendeteksi wajah dan fitur wajah secara real-time
2. Menganalisis kedipan mata dengan akurasi tinggi
3. Mendeteksi pose tubuh dan mengukur sudut sendi
4. Mengimplementasikan solusi praktis untuk analisis gerakan manusia